In [ ]:
import requests

url = "https://api-web.nhle.com/v1/standings/now"
response =requests.get(url)
print(response)
data = response.json()
data.keys()
data['standings']


<Response [200]>


dict_keys(['wildCardIndicator', 'standingsDateTimeUtc', 'standings'])

In [2]:
for team in data.get("standings", []):
    name = team["teamName"]["default"]
    abbrev = team["teamAbbrev"]["default"]
    points = team["points"]
    logo = team["teamLogo"]
    
    print(f"[{abbrev}] {name} - Points: {points} | Logo: {logo}")

[COL] Colorado Avalanche - Points: 121 | Logo: https://assets.nhle.com/logos/nhl/svg/COL_light.svg
[CAR] Carolina Hurricanes - Points: 113 | Logo: https://assets.nhle.com/logos/nhl/svg/CAR_light.svg
[DAL] Dallas Stars - Points: 112 | Logo: https://assets.nhle.com/logos/nhl/svg/DAL_light.svg
[BUF] Buffalo Sabres - Points: 109 | Logo: https://assets.nhle.com/logos/nhl/svg/BUF_light.svg
[TBL] Tampa Bay Lightning - Points: 106 | Logo: https://assets.nhle.com/logos/nhl/svg/TBL_light.svg
[MTL] Montréal Canadiens - Points: 106 | Logo: https://assets.nhle.com/logos/nhl/svg/MTL_light.svg
[MIN] Minnesota Wild - Points: 104 | Logo: https://assets.nhle.com/logos/nhl/svg/MIN_light.svg
[BOS] Boston Bruins - Points: 100 | Logo: https://assets.nhle.com/logos/nhl/svg/BOS_light.svg?season=20252026
[OTT] Ottawa Senators - Points: 99 | Logo: https://assets.nhle.com/logos/nhl/svg/OTT_light.svg
[PIT] Pittsburgh Penguins - Points: 98 | Logo: https://assets.nhle.com/logos/nhl/svg/PIT_light.svg
[PHI] Philadelp

In [18]:
import requests

url = "https://api-web.nhle.com/v1/standings/now"
response = requests.get(url)
data = response.json()

teams_list = []

for team in data.get("standings", []):
    extracted_data = {
        "team_abbrev": team["teamAbbrev"]["default"],
        "team_name": team["teamName"]["default"],
        "conference_name": team["conferenceName"],
        "division_name": team["divisionName"],
        "logo_url": team["teamLogo"]
    }
    teams_list.append(extracted_data)

# Print the extracted list of teams
for team in teams_list:
    print(team)

{'team_abbrev': 'COL', 'team_name': 'Colorado Avalanche', 'conference_name': 'Western', 'division_name': 'Central', 'logo_url': 'https://assets.nhle.com/logos/nhl/svg/COL_light.svg'}
{'team_abbrev': 'CAR', 'team_name': 'Carolina Hurricanes', 'conference_name': 'Eastern', 'division_name': 'Metropolitan', 'logo_url': 'https://assets.nhle.com/logos/nhl/svg/CAR_light.svg'}
{'team_abbrev': 'DAL', 'team_name': 'Dallas Stars', 'conference_name': 'Western', 'division_name': 'Central', 'logo_url': 'https://assets.nhle.com/logos/nhl/svg/DAL_light.svg'}
{'team_abbrev': 'BUF', 'team_name': 'Buffalo Sabres', 'conference_name': 'Eastern', 'division_name': 'Atlantic', 'logo_url': 'https://assets.nhle.com/logos/nhl/svg/BUF_light.svg'}
{'team_abbrev': 'TBL', 'team_name': 'Tampa Bay Lightning', 'conference_name': 'Eastern', 'division_name': 'Atlantic', 'logo_url': 'https://assets.nhle.com/logos/nhl/svg/TBL_light.svg'}
{'team_abbrev': 'MTL', 'team_name': 'Montréal Canadiens', 'conference_name': 'Eastern'

In [25]:
import mysql.connector
import requests

# Fetch JSON data from the API
url = "https://api-web.nhle.com/v1/standings/now"
response = requests.get(url)
data = response.json()

# MySQL connection configuration
db_config = {
    "host": "127.0.0.1",
    "user": "root",
    "password": "Manoj1297@",
    "database": "hockeyteam"
}

try:
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()

    # Insert query using ON DUPLICATE KEY UPDATE to prevent duplicate team_abbrev errors
    insert_query = """
    INSERT INTO nhl_teams (team_abbrev, team_name, conference_name, division_name, logo_url)
    VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        team_name = VALUES(team_name),
        conference_name = VALUES(conference_name),
        division_name = VALUES(division_name),
        logo_url = VALUES(logo_url);
    """

    teams_data = []
    for team in data.get("standings", []):
        teams_data.append((
            team["teamAbbrev"]["default"],
            team["teamName"]["default"],
            team["conferenceName"],
            team["divisionName"],
            team["teamLogo"]
        ))

    # Perform bulk insert
    cursor.executemany(insert_query, teams_data)
    conn.commit()

    print(f"Successfully processed {len(teams_data)} teams into MySQL.")

except mysql.connector.Error as err:
    print(f"Database Error: {err}")

finally:
    if 'cursor' in locals():
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()

Successfully processed 32 teams into MySQL.


In [27]:

import mysql.connector
import requests

# 1. Fetch JSON data from the API
url = "https://api-web.nhle.com/v1/standings/now"
response = requests.get(url)
data = response.json()

# 2. Database connection configuration
db_config = {
    "host": "127.0.0.1",
    "user": "root",
    "password": "Manoj1297@",
    "database": "hockeyteam"
}

try:
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()

    # Create a mapping dictionary of team_abbrev -> team_id from database
    cursor.execute("SELECT team_abbrev, team_id FROM nhl_teams")
    team_map = {abbrev: team_id for abbrev, team_id in cursor.fetchall()}

    insert_query = """
    INSERT INTO nhl_standings (
        team_id, season, games_played, wins, losses, ot_losses,
        points, goals_for, goals_against, home_wins, away_wins,
        streak_type, streak_count
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """

    standings_data = []
    for team in data.get("standings", []):
        abbrev = team["teamAbbrev"]["default"]
        
        # Ensure team exists in foreign table
        if abbrev not in team_map:
            print(f"Warning: Team abbreviation '{abbrev}' not found in nhl_teams table. Skipping record.")
            continue

        team_id = team_map[abbrev]
        season = str(team.get("seasonId", ""))
        games_played = team.get("gamesPlayed", 0)
        wins = team.get("wins", 0)
        losses = team.get("losses", 0)
        ot_losses = team.get("otLosses", 0)
        points = team.get("points", 0)
        goals_for = team.get("goalFor", 0)
        goals_against = team.get("goalAgainst", 0)
        home_wins = team.get("homeWins", 0)
        away_wins = team.get("roadWins", 0)
        streak_type = team.get("streakCode", "")
        streak_count = team.get("streakCount", 0)

        standings_data.append((
            team_id, season, games_played, wins, losses, ot_losses,
            points, goals_for, goals_against, home_wins, away_wins,
            streak_type, streak_count
        ))

    # Bulk insert standings data
    cursor.executemany(insert_query, standings_data)
    conn.commit()

    print(f"Successfully loaded {cursor.rowcount} standings records into MySQL.")

except mysql.connector.Error as err:
    print(f"Database Error: {err}")

finally:
    if 'cursor' in locals():
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()

Successfully loaded 32 standings records into MySQL.


In [8]:
import requests

url = "https://api-web.nhle.com/v1/club-schedule-season/COL/20252026"
response =requests.get(url)
print(response)
data = response.json()
data.keys()
#data['standings']

<Response [200]>


dict_keys(['previousSeason', 'currentSeason', 'nextSeason', 'clubTimezone', 'clubUTCOffset', 'games'])

In [4]:
import requests
import mysql.connector

# MySQL database configuration
db_config = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': 'Manoj1297@',
    'database': 'hockeyteam'
}
def sync_nhl_players():
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor(dictionary=True)

    # Fetch team_id and team_abbrev from nhl_teams table
    cursor.execute("SELECT team_id, team_abbrev FROM nhl_teams")
    teams = cursor.fetchall()

    upsert_query = """
        INSERT INTO nhl_players (
            player_id, team_id, first_name, last_name, position,
            jersey_number, birth_date, birth_country, height_cm,
            weight_kg, shoots_catches, headshot_url
        ) VALUES (
            %(player_id)s, %(team_id)s, %(first_name)s, %(last_name)s, %(position)s,
            %(jersey_number)s, %(birth_date)s, %(birth_country)s, %(height_cm)s,
            %(weight_kg)s, %(shoots_catches)s, %(headshot_url)s
        )
        ON DUPLICATE KEY UPDATE
            team_id = VALUES(team_id),
            first_name = VALUES(first_name),
            last_name = VALUES(last_name),
            position = VALUES(position),
            jersey_number = VALUES(jersey_number),
            birth_date = VALUES(birth_date),
            birth_country = VALUES(birth_country),
            height_cm = VALUES(height_cm),
            weight_kg = VALUES(weight_kg),
            shoots_catches = VALUES(shoots_catches),
            headshot_url = VALUES(headshot_url);
    """

    all_players = []

    for team in teams:
        team_id = team['team_id']
        team_abbrev = team['team_abbrev']

        url = f"https://api-web.nhle.com/v1/roster/{team_abbrev}/current"
        response = requests.get(url)

        if response.status_code != 200:
            print(f"Skipping {team_abbrev}: HTTP {response.status_code}")
            continue

        data = response.json()

        for category in ['forwards', 'defensemen', 'goalies']:
            for player in data.get(category, []):
                all_players.append({
                    'player_id': player.get('id'),
                    'team_id': team_id,
                    'first_name': player.get('firstName', {}).get('default'),
                    'last_name': player.get('lastName', {}).get('default'),
                    'position': player.get('positionCode'),
                    'jersey_number': player.get('sweaterNumber'),
                    'birth_date': player.get('birthDate'),
                    'birth_country': player.get('birthCountry'),
                    'height_cm': player.get('heightInCentimeters'),
                    'weight_kg': player.get('weightInKilograms'),
                    'shoots_catches': player.get('shootsCatches'),
                    'headshot_url': player.get('headshot')
                })

    if all_players:
        cursor.executemany(upsert_query, all_players)
        conn.commit()
        print(f"Successfully processed {len(all_players)} players into nhl_players.")

    cursor.close()
    conn.close()

if __name__ == "__main__":
    sync_nhl_players()

Successfully processed 1291 players into nhl_players.


In [12]:
import requests
import mysql.connector

db_config = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': 'Manoj1297@',
    'database': 'hockeyteam'
}

def sync_nhl_games(season="20252026"):
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor(dictionary=True)

    # 1. Fetch team abbreviations from nhl_teams
    cursor.execute("SELECT team_id, team_abbrev FROM nhl_teams")
    teams = cursor.fetchall()

    # Use INSERT IGNORE to skip duplicate game_id entries safely
    insert_query = """
        INSERT IGNORE INTO nhl_games (
            game_id, season, game_type, game_date, home_team_id,
            away_team_id, home_score, away_score, game_state, venue_name
        ) VALUES (
            %(game_id)s, %(season)s, %(game_type)s, %(game_date)s, %(home_team_id)s,
            %(away_team_id)s, %(home_score)s, %(away_score)s, %(game_state)s, %(venue_name)s
        );
    """

    all_games = {}

    # 2. Call endpoint once per team abbreviation
    for team in teams:
        team_abbrev = team['team_abbrev']
        url = f"https://api-web.nhle.com/v1/club-schedule-season/{team_abbrev}/{season}"
        response = requests.get(url)

        if response.status_code != 200:
            print(f"Skipping {team_abbrev}: HTTP {response.status_code}")
            continue

        data = response.json()

        for game in data.get('games', []):
            game_id = game.get('id')

            # Deduplicate locally before batch insertion
            if game_id not in all_games:
                all_games[game_id] = {
                    'game_id': game_id,
                    'season': str(game.get('season', season)),
                    'game_type': game.get('gameType'),
                    'game_date': game.get('gameDate'),
                    'home_team_id': game.get('homeTeam', {}).get('id'),
                    'away_team_id': game.get('awayTeam', {}).get('id'),
                    'home_score': game.get('homeTeam', {}).get('score'),
                    'away_score': game.get('awayTeam', {}).get('score'),
                    'game_state': game.get('gameState'),
                    'venue_name': game.get('venue', {}).get('default')
                }

    # 3. Insert unique game records into MySQL
    if all_games:
        cursor.executemany(insert_query, list(all_games.values()))
        conn.commit()
        print(f"Successfully processed {len(all_games)} unique games into nhl_games.")

    cursor.close()
    conn.close()

if __name__ == "__main__":
    sync_nhl_games("20252026")

Successfully processed 1498 unique games into nhl_games.


In [1]:
import json
import mysql.connector

# Database configuration
db_config = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': 'Manoj1297@',
    'database': 'hockeyteam'
}


def load_game_stats_from_json(json_filepath):
    # Connect to MySQL
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()

    # Read the JSON file
    with open(json_filepath, 'r', encoding='utf-8') as f:
        stats_data = json.load(f)

    # Prepared SQL query for inserting/upserting JSON records
    upsert_query = """
        INSERT IGNORE INTO nhl_game_stats (
            stat_id, game_id, player_id, team_id, goals, assists,
            points, shots_on_goal, penalty_min, toi, plus_minus
        ) VALUES (
            %(stat_id)s, %(game_id)s, %(player_id)s, %(team_id)s, %(goals)s, %(assists)s,
            %(points)s, %(shots_on_goal)s, %(penalty_min)s, %(toi)s, %(plus_minus)s
        )
        ON DUPLICATE KEY UPDATE
            game_id = VALUES(game_id),
            player_id = VALUES(player_id),
            team_id = VALUES(team_id),
            goals = VALUES(goals),
            assists = VALUES(assists),
            points = VALUES(points),
            shots_on_goal = VALUES(shots_on_goal),
            penalty_min = VALUES(penalty_min),
            toi = VALUES(toi),
            plus_minus = VALUES(plus_minus);
    """

    # Batch insert/update records into MySQL
    if stats_data:
        cursor.executemany(upsert_query, stats_data)
        conn.commit()
        print(f"Successfully processed {len(stats_data)} records into nhl_game_stats.")

    cursor.close()
    conn.close()

if __name__ == "__main__":
    # Update your local file path here:
    load_game_stats_from_json(r"E://AI-ML-SQL//game_stats.json")

Successfully processed 47408 records into nhl_game_stats.


In [2]:
import json
import pymysql

# 1. Database Configuration
DB_CONFIG = {
    "host": "127.0.0.1",
    "user": "root",   
    "password": "Manoj1297@", 
    "database": "hockeyteam", 
    "cursorclass": pymysql.cursors.DictCursor
}

# 2. Path to your local JSON file
JSON_FILE_PATH = "E://AI-ML-SQL//skater_season_stats.json"  # Update with full path if in a different directory

def load_json_to_database():
    connection = pymysql.connect(**DB_CONFIG)
    
    try:
        with connection.cursor() as cursor:
            # Step A: Pre-fetch all player positions from nhl_players for fast lookup
            print("Fetching player positions from nhl_players...")
            cursor.execute("SELECT player_id, position FROM nhl_players")
            player_positions = {row["player_id"]: row["position"] for row in cursor.fetchall()}

            # Step B: Read local JSON file
            print(f"Reading local JSON file: {JSON_FILE_PATH}...")
            with open(JSON_FILE_PATH, "r", encoding="utf-8") as file:
                data = json.load(file)

            skater_rows = []
            goalie_rows = []

            # Step C: Separate rows into Skaters vs Goalies
            for row in data:
                p_id = row.get("player_id")
                pos = player_positions.get(p_id)

                if pos == "G":
                    # Route to nhl_goalie_season_stats
                    goalie_rows.append((
                        row.get("stat_id"),
                        p_id,
                        row.get("season"),
                        row.get("team_id"),
                        row.get("games_played"),
                        row.get("wins", 0),
                        row.get("losses", 0),
                        row.get("ot_losses", 0),
                        row.get("save_pct", 0.0),
                        row.get("goals_against_avg", 0.0),
                        row.get("shutouts", 0),
                        row.get("saves", 0)
                    ))
                else:
                    # Route to nhl_skater_season_stats (Forwards / Defensemen)
                    skater_rows.append((
                        row.get("stat_id"),
                        p_id,
                        row.get("season"),
                        row.get("team_id"),
                        row.get("games_played"),
                        row.get("goals"),
                        row.get("assists"),
                        row.get("points"),
                        row.get("plus_minus"),
                        row.get("penalty_min"),
                        row.get("shots"),
                        row.get("avg_toi")
                    ))

            # Step D: Insert Skaters
            if skater_rows:
                skater_sql = """
                    INSERT IGNORE INTO nhl_skater_season_stats (
                        stat_id, player_id, season, team_id, games_played,
                        goals, assists, points, plus_minus, penalty_min, shots, avg_toi
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """
                cursor.executemany(skater_sql, skater_rows)
                print(f"Loaded {len(skater_rows)} records into nhl_skater_season_stats.")

            # Step E: Insert Goalies
            if goalie_rows:
                goalie_sql = """
                    INSERT IGNORE INTO nhl_goalie_season_stats (
                        stat_id, player_id, season, team_id, games_played,
                        wins, losses, ot_losses, save_pct, goals_against_avg, shutouts, saves
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """
                cursor.executemany(goalie_sql, goalie_rows)
                print(f"Loaded {len(goalie_rows)} records into nhl_goalie_season_stats.")

            connection.commit()
            print("Data migration completed successfully!")

    except Exception as e:
        connection.rollback()
        print(f"Error occurred during data migration: {e}")
    finally:
        connection.close()

if __name__ == "__main__":
    load_json_to_database()

Fetching player positions from nhl_players...
Reading local JSON file: E://AI-ML-SQL//skater_season_stats.json...
Loaded 714 records into nhl_skater_season_stats.
Data migration completed successfully!


In [3]:
import json
import pymysql

# 1. Database connection configuration
DB_CONFIG = {
    "host": "127.0.0.1",
    "user": "root",   
    "password": "Manoj1297@", 
    "database": "hockeyteam", 
    "cursorclass": pymysql.cursors.DictCursor
}

# 2. Local JSON file path
json_file_path = "E://AI-ML-SQL//goalie_season_stats.json"

def load_goalie_stats():
    # Connect to MySQL
    connection = pymysql.connect(**db_config)
    
    try:
        with connection.cursor() as cursor:
            # Load data from local JSON file
            with open(json_file_path, "r", encoding="utf-8") as file:
                data = json.load(file)

            # SQL insert statement with INSERT IGNORE to skip duplicate stat_id entries
            sql = """
                INSERT IGNORE INTO nhl_goalie_season_stats (
                    stat_id, player_id, season, team_id, games_played,
                    wins, losses, ot_losses, save_pct, goals_against_avg,
                    shutouts, saves
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """

            # Extract records from JSON
            records = []
            for item in data:
                records.append((
                    item.get("stat_id"),
                    item.get("player_id"),
                    item.get("season"),
                    item.get("team_id"),
                    item.get("games_played"),
                    item.get("wins"),
                    item.get("losses"),
                    item.get("ot_losses"),
                    item.get("save_pct"),
                    item.get("goals_against_avg"),
                    item.get("shutouts"),
                    item.get("saves")
                ))

            # Execute batch insert
            cursor.executemany(sql, records)
            connection.commit()
            print(f"Successfully inserted {cursor.rowcount} records into nhl_goalie_season_stats.")

    except Exception as e:
        connection.rollback()
        print(f"An error occurred: {e}")
    finally:
        connection.close()

if __name__ == "__main__":
    load_goalie_stats()

Successfully inserted 80 records into nhl_goalie_season_stats.
